# NVIDIA NeMo Agent Toolkit Hello World Example

In this notebook, we walk through building and running a simple workflow using NeMo Agent Toolkit.

## Table of Contents

- [0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Installing NeMo Agent Toolkit](#installing-nat)
- [1) Defining the Workflow](#defining-the-workflow)
- [2) Run The Workflow](#run-the-workflow)
- [3) Next Steps](#next-steps)

<span style="color:rgb(0, 31, 153); font-style: italic;">Note: In Google Colab use the Table of Contents tab to navigate.</span>

<a id="setup"></a>
## 0) Setup

<a id="prereqs"></a>
### 0.1) Prerequisites
- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Python Packages:** `pip`

<a id="api-keys"></a>
### 0.2) API Keys
For this notebook, you will need the following API keys to run all examples end-to-end:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys

Then you can run the cell below:

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

In [ ]:
# Suppress NAT CLI usage telemetry for this tutorial. Tutorial runs aren't
# useful analytics signal, and ``setdefault`` respects an explicit override
# (e.g. for developing the telemetry feature itself:
# ``NAT_TELEMETRY_ENABLED=true jupyter lab ...``).
os.environ.setdefault("NAT_TELEMETRY_ENABLED", "false")

<a id="installing-nat"></a>
### 0.3) Installing NeMo Agent Toolkit

NeMo Agent Toolkit can be installed through the PyPI `nvidia-nat` package, the recommended way to install `nvidia-nat` is through `uv`.

First, we will install `uv` which offers parallel downloads and faster dependency resolution.

In [ ]:
!pip install uv

There are several optional subpackages available, the `langchain` subpackage contains components needed for this notebook.

In [ ]:
%%bash
uv pip show -q "nvidia-nat-langchain"
if [ $? -ne 0 ]; then
    uv pip install "nvidia-nat[langchain]"
else
    echo "nvidia-nat[langchain] is already installed"
fi

<a id="defining-the-workflow"></a>
## 1.0) Defining the Workflow

Workflows in NeMo Agent Toolkit are defined in a YAML file. 

In [ ]:
%%writefile workflow.yml
functions:
   # Add a tool to search wikipedia
   wikipedia_search:
      _type: wiki_search
      max_results: 2

llms:
   # Tell NeMo Agent Toolkit which LLM to use for the agent
   nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3.5-lightning-30b-a3b
      temperature: 0.0
      chat_template_kwargs:
        enable_thinking: false

workflow:
   # Use an agent that 'reasons' and 'acts'
   _type: react_agent
   # Give it access to our wikipedia search tool
   tool_names: [wikipedia_search]
   # Tell it which LLM to use
   llm_name: nim_llm
   # Make it verbose
   verbose: true
   # Retry up to 3 times
   parse_agent_response_max_retries: 3

## 2.0) Run The Workflow
You can run a workflow by using `nat run` CLI command:

In [ ]:
!nat run --config_file workflow.yml --input "List five subspecies of Aardvarks"

## 3.0) Next Steps
In this notebook we skipped over many details in the pursuit of brevity, in the `getting_started_with_nat.ipynb` notebook we will discuss these details more thoroughly.